# Un Sistema per Sincronizzare i File tra Due Cartelle
## Caso d'Uso Aziendale: SyncEase
### Introduzione all'Azienda
SyncEase è una società tecnologica che sviluppa soluzioni software per migliorare la gestione e la condivisione dei dati aziendali. Una delle loro sfide principali è quella di garantire che i file critici siano sempre sincronizzati tra dispositivi e server, per evitare perdite di dati o versioni obsolete.

### Problema
Molte aziende si affidano a sistemi manuali o software non ottimizzati per mantenere sincronizzati i file tra due cartelle, ad esempio tra un server locale e uno di backup. Questo approccio è soggetto a errori, come la perdita di file aggiornati o la duplicazione di dati. Inoltre, il processo può essere lento, specialmente quando il volume di dati è elevato.

### Obiettivo del Progetto
L'obiettivo è sviluppare un sistema software che utilizzi il multiprocessing e il multithreading in Python per sincronizzare in modo efficiente i file tra due cartelle. Questo sistema dovrà:

- Copiare i file nuovi o modificati dalla cartella sorgente a quella di destinazione.
- Eliminare i file obsoleti nella cartella di destinazione se non esistono più nella sorgente.
- Gestire grandi volumi di file in modo rapido ed efficiente, sfruttando al meglio le risorse hardware.
### Benefici Attesi
- Riduzione degli errori: Sincronizzazione affidabile e automatica senza intervento manuale.
- Ottimizzazione delle prestazioni: Utilizzo efficiente di CPU e I/O grazie al multiprocessing e al multithreading.
- Miglioramento della sicurezza: Backup sempre aggiornati e consistenti tra le due cartelle.
### Specifiche del Progetto
- Input: Due percorsi di cartelle: source_folder e destination_folder.
- Output: La cartella di destinazione è sincronizzata con quella sorgente.
### Funzionalità Chiave:
- Utilizzo di multithreading per accelerare la lettura e la scrittura dei file.
- Utilizzo di multiprocessing per gestire in parallelo sottogruppi di file.
Verifica di:
- File nuovi: Copia solo i file che non esistono nella cartella di destinazione.
- File modificati: Aggiorna i file nella destinazione se quelli nella sorgente sono più recenti.
- File obsoleti: Rimuove i file nella destinazione che non esistono più nella sorgente.
### Consegna
Scrivi un programma Python che implementi il sistema di sincronizzazione dei file. Il tuo codice deve essere ben documentato, con commenti che spieghino chiaramente le scelte implementative e il flusso del programma. Testa il suo funzionamento con almeno 3 casi reali. Per esempio: - Una cartella con pochi file. - Una cartella con molti file. - Una cartella che include sottocartelle.



## Import e configurazione logging

Importiamo le librerie necessarie e configuriamo il logger per tracciare ogni operazione con timestamp e livello di severità.

In [ ]:
from pathlib import Path
import shutil
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed, ProcessPoolExecutor
from typing import List
import numpy as np
import random
import time


# Configurazione logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True
)
logger = logging.getLogger(__name__)

# Percorsi cartelle
src = Path("/content/src")
dst = Path("/content/dst")

print(f"Sorgente:     {src}")
print(f"Destinazione: {dst}")

## Setup file di test

In [ ]:
def setup_file(src: Path, n: int = 4):
    """
    Crea n file di testo nella cartella sorgente.

    Args:
        src (Path): cartella sorgente in cui creare i file.
        n (int): numero di file da creare. Di default n=4
    """
    src.mkdir(parents=True, exist_ok=True)
    for i in range(1, n + 1):
        file = src / f"file{i}.txt"
        if not file.exists():
            file.write_text(f"Contenuto del file {i}")
            logger.debug(f"Creato: {file}")
        else:
            logger.debug(f"Già esistente, saltato: {file}")


# Esecuzione setup
setup_file(src, n=4)
print(f"\nFile in src: {[f.name for f in src.iterdir()]}")

## Funzioni operative

Definisce le due operazioni elementari eseguite dai thread:
- `copia_file` → copia un singolo file da src a dst preservando i metadati
- `elimina_file` → elimina un singolo file da dst

Ogni funzione agisce solo su un file


In [ ]:
def copia_file(src: Path, dst: Path, relpath: Path) -> str:
    """
    Copia un singolo file da src a dst mantenendo la struttura di sottocartelle.

    Utilizziamo shutil.copy2 per preservare i metadati (data modifica, size).
    Questo è fondamentale per il confronto dei file fra le due cartelle

    Args:
        src (Path): cartella sorgente.
        dst (Path): cartella destinazione.
        relpath (Path): percorso relativo del file rispetto a src.

    Returns:
        str: messaggio descrittivo dell'operazione.

    Raises:
        OSError: se la copia fallisce per permessi o disco pieno.
    """

    dst_path = dst / relpath
    dst_path.parent.mkdir(parents=True, exist_ok=True)  # crea sottocartelle se necessario
    shutil.copy2(src / relpath, dst_path)
    return f"Copiato: {relpath}"


def elimina_file(dst: Path, relpath: Path) -> str:
    """
    Elimina un singolo file dalla cartella destinazione.

    Args:
        dst (Path): cartella destinazione.
        relpath (Path): percorso relativo del file da eliminare.

    Returns:
        str: messaggio descrittivo dell'operazione.

    Raises:
        FileNotFoundError: se il file non esiste al momento dell'eliminazione.
        OSError: se l'eliminazione fallisce per problemi di permessi.
    """
    (dst / relpath).unlink()
    return f"Eliminato: {relpath}"


## Scansione cartelle

La funzione `scansione()` percorre ricorsivamente una cartella e raccoglie per ogni file:
- `st_size` → dimensione in byte
- `st_mtime` → timestamp dell'ultima modifica

Questi metadati vengono usati da `sync()` per decidere se un file va copiato o è già aggiornato.

In [ ]:
def scansione(cartella: Path) -> dict[Path, list]:
    """
    Scansiona ricorsivamente una cartella e restituisce i metadati di ogni file.

    Args:
        cartella (Path): cartella da scansionare.

    Returns:
        dict[Path, list]: dizionario nella forma
            { Path('sottocartella/file.txt'): [dimensione_bytes, timestamp] }
    """
    risultato = {}
    for file in cartella.rglob("*"):
        if file.is_file():
            info = file.stat()
            risultato[file.relative_to(cartella)] = [info.st_size, info.st_mtime]
    return risultato


# Anteprima della scansione di src
print("Scansione di src:")
for path, meta in scansione(src).items():
    print(f"  {path} → size={meta[0]}B, mtime={meta[1]:.0f}")

Creiamo delle funzioni di copia e di eliminazione di sottogruppi di file utilizzando ThreadPoolExecutor

In [ ]:
def copia_gruppo(src: Path, dst: Path, da_copiare: list[Path]) -> list:
    """
    Copia un sottogruppo di file da src a dst usando ThreadPoolExecutor.
    Verrà eseguita da ogni processo figlio lanciato da ProcessPoolExecutor.

    Args:
        src (Path): cartella sorgente.
        dst (Path): cartella destinazione.
        da_copiare (list[Path]): sottogruppo di file da copiare.

    Returns:
        list: messaggi di log da restituire al processo principale.
    """
    risultati = []
    with ThreadPoolExecutor() as executor:
        futures_copia = {
            executor.submit(copia_file, src, dst, k): k
            for k in da_copiare
        }
        for future in as_completed(futures_copia):
            try:
                risultati.append(future.result())
            except Exception as e:
                risultati.append(f"ERRORE: {futures_copia[future]}: {e}")
    return risultati

def elimina_gruppo(dst: Path, da_eliminare: list[Path]) -> list:
    """
    Elimina un gruppo di file da dst usando ThreadPoolExecutor.
    Verrà eseguita dal processo principale.

    Args:
        dst (Path): cartella destinazione.


        da_eliminare (list[Path]): file da eliminare.

    Returns:
        list: messaggi di log.
    """
    risultati = []
    with ThreadPoolExecutor() as executor:
        futures_elimina = {
            executor.submit(elimina_file, dst, k): k
            for k in da_eliminare
        }
        for future in as_completed(futures_elimina):
            try:
                risultati.append(future.result())
            except Exception as e:
                risultati.append(f"ERRORE: {futures_elimina[future]}: {e}")
    return risultati




In [ ]:
def sync_multithread_copia(src: Path, dst: Path, modifiche: List) -> None:
    """
    Distribuisce i sottogruppi di file su più processi con ProcessPoolExecutor.
    Ogni processo usa internamente ThreadPoolExecutor per le copie.

    Args:
        src (Path): cartella sorgente.
        dst (Path): cartella destinazione.
        modifiche (List): lista di sottogruppi di file da copiare.
    """
    with ProcessPoolExecutor() as executor:
        futures = {
            executor.submit(copia_gruppo, src, dst, modifiche[i]): i
            for i in range(len(modifiche))
        }
        for future in as_completed(futures):
            try:
                for msg in future.result():
                    print(msg, flush=True)
            except Exception as e:
                print(f"ERRORE gruppo {futures[future]}: {e}", flush=True)

Uniamo tutti gli step nella funzione sync finale

In [ ]:

def sync(src: Path, dst: Path) -> None:
    """
    Sincronizza dst con src in modo concorrente.

    Copia i file nuovi o modificati da src a dst ed elimina da dst
    i file non più presenti in src.
    Le copie vengono distribuite su più processi tramite ProcessPoolExecutor,
    ognuno dei quali usa ThreadPoolExecutor per le operazioni I/O.
    Le eliminazioni vengono gestite dal processo principale con ThreadPoolExecutor.

    Args:
        src (Path): cartella sorgente (riferimento).
        dst (Path): cartella destinazione (viene allineata a src).
    """
    # creiamo sorgente e destinazione nel caso in cui non esistano già
    src.mkdir(parents=True, exist_ok=True)
    dst.mkdir(parents=True, exist_ok=True)

    # --- Fase 1: Scansione ---
    log_src = scansione(src)
    log_dst = scansione(dst)

    logger.info(f"File in sorgente:     {len(log_src)}")
    logger.info(f"File in destinazione: {len(log_dst)}")

    # --- Fase 2: Decisione ---
    da_copiare = []
    for k, v in log_src.items():
        if k not in log_dst:
            logger.info(f"  [NUOVO]      {k}")
            da_copiare.append(k)
        elif v[1] > log_dst[k][1] or v[0] != log_dst[k][0]: #verifichiamo variazioni sul timestamp o sulle dimensioni
            logger.info(f"  [MODIFICATO] {k}")
            da_copiare.append(k)
        else:
            logger.debug(f"  [INVARIATO]  {k}")

    da_eliminare = [k for k in log_dst if k not in log_src]
    for k in da_eliminare:
        logger.info(f"  [OBSOLETO]   {k}")

    if not da_copiare and not da_eliminare:
        logger.info("Nessuna operazione necessaria — cartelle già sincronizzate.")
        return

    # --- Fase 3: Esecuzione ---

    # divide da_copiare in 5 sottogruppi e li distribuisce su 5 processi
    gruppi = np.array_split(da_copiare, 5)
    sync_multithread_copia(src, dst, gruppi)

    # le eliminazioni vengono gestite con il processo principale
    for msg in elimina_gruppo(dst, da_eliminare):
        print(msg, flush=True)

    logger.info("Sincronizzazione completata.")

## Esecuzione e verifica

Eseguiamo la sincronizzazione e verifichiamo che `dst` contenga gli stessi file di `src`.

In [ ]:
def pulisci_cartelle(src: Path, dst: Path):
    for cartella in [src, dst]:
        if cartella.exists():
            shutil.rmtree(cartella)
        cartella.mkdir(parents=True)
    logger.info("Cartelle pulite")  # ← era print()

def esegui_test(n: int):
    pulisci_cartelle(src, dst)
    setup_file(src, n=n)
    logger.info(f"Test con {n} file")  # ← era print()
    inizio = time.time()
    sync(src, dst)
    durata = time.time() - inizio
    logger.info(f"Durata: {durata:.2f}s")  # ← era print()

In [ ]:
def test_function(n: int, k: int = 5):
    """
    Elimina e modifica file casuali in src
    Args:
        n: numero totale di file in src
        k: quanti file eliminare/modificare
    """
    tutti_i_file = [src / f"file{i}.txt" for i in range(1, n + 1)]

    # elimina k file casuali da src
    da_eliminare = random.sample(tutti_i_file, k=k)
    for file in da_eliminare:
        file.unlink()
        logger.info(f"Rimosso da src: {file.name}")

    # modifica k file casuali tra quelli rimasti
    rimasti = [f for f in tutti_i_file if f not in da_eliminare]
    da_modificare = random.sample(rimasti, k=k)
    time.sleep(0.1)
    for file in da_modificare:
        file.write_text("Contenuto modificato!")
        logger.info(f"Modificato: {file.name}")

    logger.info(f"Seconda sync — deve eliminare {k} file e aggiornare {k} file")
    sync(src, dst)


In [ ]:
def test_sottocartelle():
    pulisci_cartelle(src, dst)

    # crea struttura con sottocartelle direttamente
    (src / "subdir1").mkdir(parents=True, exist_ok=True)
    (src / "subdir2").mkdir(parents=True, exist_ok=True)
    (src / "subdir1" / "subdir3").mkdir(parents=True, exist_ok=True)

    (src / "file1.txt").write_text("root")
    (src / "subdir1" / "file1.txt").write_text("subdir1")
    (src / "subdir2" / "file1.txt").write_text("subdir2")
    (src / "subdir1" / "subdir3" / "file1.txt").write_text("subdir3")

    sync(src, dst)


### Test con 10 file

In [ ]:
esegui_test(n=10)

In [ ]:
sync(src, dst)

In [ ]:
test_function(5,2)

## Test con sottocartelle


In [ ]:
test_sottocartelle()

### Test con 500 file

In [ ]:
esegui_test(n=500)

In [ ]:
test_function(70, 10)